# Shared Configuration

This notebook is `%run` by the other example notebooks. Update the values below for your environment.

**Do not run this notebook directly** — it is included by the others via `%run ./_config`.

In [ ]:
# ---------------------------------------------------------------------------
# Configuration — UPDATE THESE for your environment
# ---------------------------------------------------------------------------
HOST = "your-hostname.com"

# Secrets — loaded from Databricks secret scope
# Set up with:
#   databricks secrets create-scope data-api
#   databricks secrets put-secret data-api api-key --string-value "YOUR_SECRET_KEY"
#   databricks secrets put-secret data-api kafka-user --string-value "YOUR_KAFKA_USER"
#   databricks secrets put-secret data-api kafka-password --string-value "YOUR_KAFKA_PASSWORD"
#   databricks secrets put-secret data-api neo4j-password --string-value "YOUR_NEO4J_PASSWORD"
#   databricks secrets put-secret data-api postgres-password --string-value "YOUR_POSTGRES_PASSWORD"

API_KEY           = dbutils.secrets.get(scope="data-api", key="api-key")
KAFKA_USER        = dbutils.secrets.get(scope="data-api", key="kafka-user")
KAFKA_PASSWORD    = dbutils.secrets.get(scope="data-api", key="kafka-password")
NEO4J_USER        = "neo4j"
NEO4J_PASSWORD    = dbutils.secrets.get(scope="data-api", key="neo4j-password")
POSTGRES_USER     = "postgres"
POSTGRES_PASSWORD = dbutils.secrets.get(scope="data-api", key="postgres-password")

# Derived URLs
API_URL         = f"https://{HOST}/api/v1"
KAFKA_BROKER    = f"{HOST}:9093"
NEO4J_URI       = f"bolt+s://{HOST}:7688"
POSTGRES_JDBC   = f"jdbc:postgresql://{HOST}:15433/datacollector?ssl=true&sslmode=require"
HEADERS         = {"X-Api-Key": API_KEY}

# Checkpoint base path — update to your catalog/schema
CHECKPOINT_BASE = "/Volumes/your_catalog/your_schema/checkpoints"

# SLED use cases
SLED_USE_CASES = [
    "student_enrollment", "grant_budget", "citizen_services",
    "k12_early_warning", "procurement", "case_management",
]

# Kafka connection options (reusable)
KAFKA_OPTIONS = {
    "kafka.bootstrap.servers": KAFKA_BROKER,
    "kafka.security.protocol": "SASL_SSL",
    "kafka.sasl.mechanism": "PLAIN",
    "kafka.sasl.jaas.config": (
        'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required '
        f'username="{KAFKA_USER}" '
        f'password="{KAFKA_PASSWORD}";'
    ),
    "startingOffsets": "earliest",
}

In [ ]:
# ---------------------------------------------------------------------------
# Helper functions
# ---------------------------------------------------------------------------
import requests
import builtins
from pyspark.sql.types import *
from pyspark.sql.functions import *


def read_kafka_stream(topic, schema):
    """Read a Kafka topic as a parsed Structured Streaming DataFrame."""
    raw = spark.readStream.format("kafka")
    for k, v in KAFKA_OPTIONS.items():
        raw = raw.option(k, v)
    raw = raw.option("subscribe", topic).load()
    return (
        raw
        .select(from_json(col("value").cast("string"), schema).alias("data"))
        .select("data.*")
    )


def read_pg_table(table_name):
    """Read a table from PostgreSQL via JDBC."""
    return (
        spark.read
        .format("jdbc")
        .option("url", POSTGRES_JDBC)
        .option("dbtable", table_name)
        .option("user", POSTGRES_USER)
        .option("password", POSTGRES_PASSWORD)
        .option("driver", "org.postgresql.Driver")
        .load()
    )


print(f"Config loaded — API: {API_URL}, Kafka: {KAFKA_BROKER}")